In [1]:
import os
import time
import requests
import pandas as pd
from datetime import date
from dotenv import load_dotenv

load_dotenv() 
print ("Libraries imported")

Libraries imported


In [10]:
import sqlite3 as sql
from datetime import date, timedelta

In [12]:
connection = sql.connect("../UAS_Tracker.db")

In [11]:
today_last_week = date.today() - timedelta(days=7)
today_last_week_str = today_last_week.isoformat()

In [15]:
sql_string = """ SELECT api_internal_id
                FROM contracts
                WHERE last_modified_date >= ?
                             """

In [16]:
cursor = connection.cursor()

In [27]:
cursor.execute( sql_string, (today_last_week_str,) )

awards = cursor.fetchall()
print(awards)

[('CONT_AWD_W900KK26FA047_9700_W900KK22D0010_9700',), ('CONT_AWD_M6700126P0036_9700_-NONE-_-NONE-',), ('CONT_AWD_FA813626F0018_9700_47QSMA19D08Q1_4732',), ('CONT_AWD_70B06C25P00000507_7014_-NONE-_-NONE-',), ('CONT_AWD_70B04C26F00000964_7014_70B02C26A00000005_7014',)]


**Endpoint documentation found here** <br>
https://github.com/fedspendingtransparency/usaspending-api/blob/master/usaspending_api/api_contracts/contracts/v2/transactions.md

In [44]:
sql_string = """INSERT OR REPLACE INTO contract_transactions 
                (transaction_id,
                type,
                action_type,
                action_date,
                description,
                modification_number,
                federal_action_obligation,
                date_added,
                api_internal_id
                )

                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                 """

In [34]:
today = date.today()

In [45]:
for award in awards:
    request = requests.post(
    url = "https://api.usaspending.gov/api/v2/transactions/",
    json = {
        "award_id": award[0],
        "page": 1,
        "sort": "action_date",
        "order": "asc",    
        "limit": 250
    }
    )
    results = request.json()["results"]
    for result in results:
        cursor.execute (sql_string, (
        result["id"],
        result["type_description"],
        result["action_type_description"],
        result["action_date"],
        result["description"],
        result["modification_number"],
        result["federal_action_obligation"],
        today.isoformat(),
        award[0],
    ))
connection.commit()